In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('parkinsons.data')
data.head(10)
X = data.drop(columns=['name', 'status'])
y = data['status']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [3]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Логистическая регрессия
logreg = LogisticRegression(max_iter=1000)
scores_log = cross_val_score(logreg, X_scaled, y, cv=5, scoring='accuracy')
print("Accuracy LogisticRegression:", scores_log.mean())

# Случайный лес
rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores_rf = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
print("Accuracy RandomForest:", scores_rf.mean())

# SVM с RBF-ядером
svc = SVC(kernel='rbf', gamma='scale')
scores_svc = cross_val_score(svc, X_scaled, y, cv=5, scoring='accuracy')
print("Accuracy SVC:", scores_svc.mean())


Accuracy LogisticRegression: 0.8102564102564103
Accuracy RandomForest: 0.7846153846153847
Accuracy SVC: 0.8461538461538461


По точности мы видим что самый лучшым вариантом будет **Метод опорных векторов** 

In [4]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [5]:
data = pd.read_csv('parkinsons.data')
def get_subject_id(name):
    parts = name.split('_')
    if len(parts) >= 3:
        return f"{parts[1]}_{parts[2]}"
    return name

data['subject_id'] = data['name'].apply(get_subject_id)
print("Уникальных субъектов:", data['subject_id'].nunique())
print("Записей на субъект (пример первых 10):")
print(data['subject_id'].value_counts().head(10))

Уникальных субъектов: 32
Записей на субъект (пример первых 10):
subject_id
R01_S35    7
R01_S27    7
R01_S21    7
R01_S01    6
R01_S25    6
R01_S49    6
R01_S44    6
R01_S43    6
R01_S42    6
R01_S39    6
Name: count, dtype: int64


In [6]:
X = data.drop(columns=['name', 'status', 'subject_id'])
y = data['status'].values

subject_status = data.groupby('subject_id')['status'].first()
unique_subjects = subject_status.index.values
subject_labels = subject_status.values

print("Признаков:", X.shape[1])

Признаков: 22


In [7]:
train_subs, test_subs = train_test_split(
    unique_subjects,
    test_size=0.2,
    random_state=42,
    stratify=subject_labels
)

train_idx = data['subject_id'].isin(train_subs)
test_idx = data['subject_id'].isin(test_subs)

X_train = X[train_idx].values
y_train = y[train_idx]
groups_train = data['subject_id'][train_idx].values

X_test = X[test_idx].values
y_test = y[test_idx]
groups_test = data['subject_id'][test_idx].values

print("Train samples:", X_train.shape[0], " Test samples:", X_test.shape[0])
print("Train subjects:", len(train_subs), " Test subjects:", len(test_subs))

Train samples: 152  Test samples: 43
Train subjects: 25  Test subjects: 7


In [8]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC())
])
param_grid = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    'svc__kernel': ['rbf']
}

gk = GroupKFold(n_splits=5)

grid = GridSearchCV(pipe, param_grid, cv=gk, n_jobs=-1, scoring='accuracy', verbose=1)

grid.fit(X_train, y_train, groups=groups_train)

print("Лучшие параметры:", grid.best_params_)
print("Лучшая CV accuracy (на train):", grid.best_score_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Лучшие параметры: {'svc__C': 100, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'}
Лучшая CV accuracy (на train): 0.8146236559139786


In [9]:
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {test_acc:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

Test accuracy: 0.8372

Confusion matrix:
[[ 5  7]
 [ 0 31]]

Classification report:
              precision    recall  f1-score   support

           0       1.00      0.42      0.59        12
           1       0.82      1.00      0.90        31

    accuracy                           0.84        43
   macro avg       0.91      0.71      0.74        43
weighted avg       0.87      0.84      0.81        43



In [10]:

subject_status = data.groupby('subject_id')['status'].first()
pos_subs = subject_status[subject_status == 1].index.values
neg_subs = subject_status[subject_status == 0].index.values
n_pos_total = len(pos_subs)
n_neg_total = len(neg_subs)
total_subs = n_pos_total + n_neg_total

print(f"Всего субъектов: {total_subs}, больных: {n_pos_total}, здоровых: {n_neg_total}")

# Параметры
train_fracs = np.linspace(0.2, 1.0, 6)
train_scores = []
val_scores = []
n_repeats = 5
rng = np.random.default_rng(42)

# Получаем параметры SVC из grid (если grid ещё существует)
best_params = grid.best_params_
svc_params = {k.split('__')[1]: v for k, v in best_params.items() if k.startswith('svc__')}

for frac in train_fracs:
    accs_train = []
    accs_val = []
    # Целевое число субъектов в train (но ниже мы будем снижать, если невозможно)
    n_target = max(2, int(round(total_subs * frac)))

    for _ in range(n_repeats):
        success = False
        # Попробуем несколько попыток подобрать допустимое разбиение
        for attempt in range(100):
            # если n_target слишком большой, уменьшаем его до (total_subs - 1) чтобы в val остался хотя бы 1 субъект
            n_train_subs = min(n_target, total_subs - 1)
            # Найдём приемлемый диапазон для количества положительных субъектов в train:
            # n_pos must be between 1 and min(n_pos_total, n_train_subs-1)
            min_n_pos = 1
            max_n_pos = min(n_pos_total, n_train_subs - 1)  # оставляем минимум 1 для негативных
            if max_n_pos < min_n_pos:
                # Невозможно взять оба класса при текущем n_train_subs -> уменьшаем n_train_subs и пробуем снова
                n_target = max(2, n_train_subs - 1)
                continue

            # Выбираем n_pos случайно в допустимом диапазоне
            n_pos = rng.integers(min_n_pos, max_n_pos + 1)
            n_neg = n_train_subs - n_pos

            # Проверка: n_neg не должен превышать общего количества негативных субъектов
            if n_neg < 1 or n_neg > n_neg_total:
                # Попробуем адаптировать n_pos так, чтобы n_neg влезал в available negatives
                # вычислим допустимый диапазон для n_pos с учётом n_neg_total
                min_n_pos2 = max(min_n_pos, n_train_subs - n_neg_total)
                max_n_pos2 = min(max_n_pos, n_train_subs - 1)
                if min_n_pos2 > max_n_pos2:
                    # уменьшим n_train_subs и попробуем заново
                    n_target = max(2, n_train_subs - 1)
                    continue
                n_pos = rng.integers(min_n_pos2, max_n_pos2 + 1)
                n_neg = n_train_subs - n_pos

            # Теперь n_pos <= n_pos_total и n_neg <= n_neg_total гарантировано


Всего субъектов: 32, больных: 24, здоровых: 8


In [11]:

cv_results = cross_validate(grid.best_estimator_, X_train, y_train, cv=gk, groups=groups_train, scoring='accuracy', return_train_score=True)
print("CV train accuracy mean:", cv_results['train_score'].mean())
print("CV val accuracy mean:", cv_results['test_score'].mean())


CV train accuracy mean: 0.8997019374068553
CV val accuracy mean: 0.8146236559139786
